In [4]:
import os
import pdfplumber
import pandas as pd

DATASET_PATH = "/content/drive/MyDrive/dataset"

data = []

for category in os.listdir(DATASET_PATH):

    category_path = os.path.join(DATASET_PATH, category)

    if not os.path.isdir(category_path):
        continue

    for pdf_file in os.listdir(category_path):

        if not pdf_file.endswith(".pdf"):
            continue

        pdf_path = os.path.join(category_path, pdf_file)

        try:

            text = ""

            with pdfplumber.open(pdf_path) as pdf:

                for page in pdf.pages[:30]:   # first 30 pages enough

                    page_text = page.extract_text()

                    if page_text:
                        text += page_text + " "

            data.append({
                "text": text,
                "label": category
            })

        except Exception as e:
            print(f"Error: {pdf_path}")
            print(e)

df = pd.DataFrame(data)

print(df.head())
print(df['label'].value_counts())

                                                text          label
0  FEAST INSTALLATION GUIDE ii Installation of FE...  Documentation
1  UG10156\nAndroid User's Guide\nRev. android-16...  Documentation
2  O R ®\nPEN ULES\nOpen Source Business\nDecisio...  Documentation
3  API doc um ent ation\nParis Mou Data Exchange ...  Documentation
4  Cloud, Lean and Complete LMS with an Emphasis\...  Documentation
label
Documentation     39
Resume            39
research_paper    39
question_paper    39
medical_report    35
Name: count, dtype: int64


In [5]:
df

,text,label
0,FEAST INSTALLATION GUIDE ii Installation of FE...,Documentation
1,UG10156\nAndroid User's Guide\nRev. android-16...,Documentation
2,O R ®\nPEN ULES\nOpen Source Business\nDecisio...,Documentation
3,API doc um ent ation\nParis Mou Data Exchange ...,Documentation
4,"Cloud, Lean and Complete LMS with an Emphasis\...",Documentation
...,...,...
186,JEE (Advanced) 2026 Paper 1\nSECTION 1 (Maximu...,question_paper
187,"PARAMEDICAL BOARD, BENGALURU\nSUB: PHYSICS (1S...",question_paper
188,"Join Telegram Group ""HaryanaJobs.in""\nHaryana ...",question_paper
189,OUTCOME BASED CURRICULUM-2016\nWorkshop\nOn\nQ...,question_paper


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

X = df["text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LinearSVC()

model.fit(X_train_tfidf, y_train)

predictions = model.predict(X_test_tfidf)

print("Accuracy :", accuracy_score(y_test, predictions))

print("\nClassification Report\n")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, predictions))

Accuracy : 0.9743589743589743

Classification Report

                precision    recall  f1-score   support

 Documentation       1.00      0.88      0.93         8
        Resume       1.00      1.00      1.00         8
medical_report       1.00      1.00      1.00         7
question_paper       1.00      1.00      1.00         8
research_paper       0.89      1.00      0.94         8

      accuracy                           0.97        39
     macro avg       0.98      0.97      0.97        39
  weighted avg       0.98      0.97      0.97        39


Confusion Matrix

[[7 0 0 0 1]
 [0 8 0 0 0]
 [0 0 7 0 0]
 [0 0 0 8 0]
 [0 0 0 0 8]]


In [13]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    vectorizer.fit_transform(X),
    y,
    cv=5
)

print(scores)
print(scores.mean())


[0.94871795 0.86842105 0.89473684 0.92105263 0.89473684]
0.9055330634278003


In [15]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Saved Successfully")

Saved Successfully
